In [0]:
from pyspark.sql.functions import *

SOURCE_TABLE = "fuel_project_dev.silver.fuel_transactions"
RATES_TABLE = "fuel_project_dev.silver.fuel_rates"
TARGET_TABLE = "fuel_project_dev.gold.fact_fuel_transactions"
CHECKPOINT_LOCATION = "/Volumes/fuel_project_dev/checkpoints/gold/fact_fuel_transactions"


In [0]:
# Create target table if not exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        fill_id STRING,
        station_id STRING,
        date_key STRING,
        hour_key INT,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        transaction_duration_sec BIGINT,
        fuel_type STRING,
        fuel_volume DOUBLE,
        payment_type STRING,
        fuel_cost DOUBLE,
        revenue DOUBLE,
        processed_at TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (station_id, date_key)
    TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact = true
    )
""")

In [0]:
# Read streaming data
transactions_stream = (
    spark.readStream
    .format("delta")
    .table(SOURCE_TABLE)
)

# Read rates as static (or streaming if rates also stream)
rates_df = spark.table(RATES_TABLE)


In [0]:

# Transform and join
fact_stream = (
    transactions_stream.alias("t")
    .join(
        rates_df.alias("r"),
        (col("t.station_id") == col("r.fuel_station_id")) &
        (col("t.fuel_type") == col("r.fuel_type")) &
        (col("t.start_time") >= col("r.start_datetime")) &
        (col("t.start_time") < col("r.end_datetime")),
        "left"
    )
    .select(
        col("t.fill_id"),
        col("t.station_id"),
        date_format(col("t.start_time"), "yyyyMMdd").alias("date_key"),
        hour(col("t.start_time")).alias("hour_key"),
        col("t.start_time"),
        col("t.end_time"),
        (unix_timestamp(col("t.end_time")) - unix_timestamp(col("t.start_time"))).alias("transaction_duration_sec"),
        col("t.fuel_type"),
        col("t.fuel_volume"),
        col("t.payment_type"),
        col("r.fuel_cost"),
        (col("t.fuel_volume") * col("r.fuel_cost")).alias("revenue"),
        current_timestamp().alias("processed_at")
    )
)


In [0]:

# Write stream
query = (
    fact_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .table(TARGET_TABLE)
)

query.awaitTermination()